# 🩺 TrOCR Fine-Tuning on RxHandBD — Handwritten Prescription OCR
**Model:** `microsoft/trocr-base-handwritten` (VisionEncoderDecoderModel)  
**Dataset:** RxHandBD — 5 578 handwritten prescription word images  
**Method:** LoRA (PEFT) · Mixed-precision fp16 · Single GPU (T4 / P100)

---
### Pipeline Overview
1. Install dependencies  
2. Dataset paths & CSV loading  
3. Data exploration & visualization  
4. TrOCR processor + model  
5. LoRA setup  
6. Dataset class & DataLoaders  
7. Training loop  
8. Evaluation (CER / WER)  
9. Inference & visualization  
10. Save model


## 📦 Step 1 — Install Dependencies

In [2]:
# Run once; restart runtime afterwards if on Colab
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = "✅" if r.returncode == 0 else "❌"
    print(f"{status}  {cmd[:70]}")
    if r.returncode != 0:
        print(r.stderr[-300:])

run("pip install -q transformers>=4.40.0 accelerate>=0.26.0")
run("pip install -q peft>=0.9.0")
run("pip install -q jiwer Pillow matplotlib seaborn tqdm pandas scikit-learn")
print("\n🚀 Done.")


✅  pip install -q transformers>=4.40.0 accelerate>=0.26.0
✅  pip install -q peft>=0.9.0
✅  pip install -q jiwer Pillow matplotlib seaborn tqdm pandas scikit-lear

🚀 Done.


In [3]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


PyTorch : 2.10.0+cpu
CUDA    : False


## 📁 Step 2 — Dataset Paths & CSV Loading
**Expected layout after extracting RxHandBD-ML.zip:**
```
rxhandbd/
└── RxHandBD-ML/
    ├── Train_Label.csv   (columns: filename, label)
    ├── Test_Label.csv
    ├── Train_Set/        (*.jpg / *.png)
    └── Test_Set/
```
Adjust `DATA_DIR` below if your layout differs.


In [ ]:
import os
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path("/content")          # change to /kaggle/working on Kaggle
DATA_DIR    = BASE_DIR / "rxhandbd"
MODEL_DIR   = BASE_DIR / "trocr_finetuned"
RESULTS_DIR = BASE_DIR / "results"

for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Define TRAIN_DIR and TEST_DIR
TRAIN_DIR = DATA_DIR / "RxHandBD-ML" / "Train_Set"
TEST_DIR  = DATA_DIR / "RxHandBD-ML" / "Test_Set"

# Create folders
for d in [DATA_DIR, TRAIN_DIR, TEST_DIR, MODEL_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📁 BASE_DIR    : {BASE_DIR}")
print(f"📁 DATA_DIR    : {DATA_DIR}")
print(f"📁 TRAIN_DIR   : {TRAIN_DIR}")
print(f"📁 TEST_DIR    : {TEST_DIR}")
print(f"📁 MODEL_DIR   : {MODEL_DIR}")
print(f"📁 RESULTS_DIR : {RESULTS_DIR}")


# ============================================================
# STEP 2: Download dataset from Mendeley
# ============================================================

from google.colab import files

files.upload()


# ============================================================
# STEP 3: Extract dataset
# ============================================================

import zipfile
from pathlib import Path

# FIX: Update ZIP_PATH to match the uploaded filename
ZIP_PATH = Path("/content/RxHandBD-ML.zip")
DATA_DIR = Path("/content/rxhandbd")

DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DATA_DIR)

print("✅ Dataset extracted!")


# ============================================================
# STEP 4: Show extracted files
# ============================================================

print("\n📂 Dataset contents:")

for item in sorted(DATA_DIR.iterdir()):
    print(" -", item.name)

📁 BASE_DIR    : /content
📁 DATA_DIR    : /content/rxhandbd
📁 TRAIN_DIR   : /content/rxhandbd/RxHandBD-ML/Train_Set
📁 TEST_DIR    : /content/rxhandbd/RxHandBD-ML/Test_Set
📁 MODEL_DIR   : /content/trocr_finetuned
📁 RESULTS_DIR : /content/results


In [ ]:
# ============================================================
# STEP 2C: Locate train/test folders and CSV labels
# The AI-compatible version has pre-organized train/test split
# ============================================================
import glob

# Find CSV files
# FIX: Update glob pattern to match actual CSV file names and location
train_csvs = list(DATA_DIR.glob("**/RxHandBD-ML/Train_Label.csv"))
test_csvs  = list(DATA_DIR.glob("**/RxHandBD-ML/Test_Label.csv"))

# Find image folders
train_folders = list(DATA_DIR.glob("**/RxHandBD-ML/Train_Set"))
test_folders  = list(DATA_DIR.glob("**/RxHandBD-ML/Test_Set"))

print("Train CSVs found:", train_csvs)
print("Test CSVs found:", test_csvs)
print("Train folders:", train_folders)
print("Test folders:", test_folders)

# Set paths
TRAIN_CSV = train_csvs[0] if train_csvs else None
TEST_CSV  = test_csvs[0]  if test_csvs  else None
TRAIN_IMG_DIR = train_folders[0] if train_folders else DATA_DIR / "train"
TEST_IMG_DIR  = test_folders[0]  if test_folders  else DATA_DIR / "test"

# Count images
train_images = list(TRAIN_IMG_DIR.glob("*.jpg")) + list(TRAIN_IMG_DIR.glob("*.png"))
test_images  = list(TEST_IMG_DIR.glob("*.jpg"))  + list(TEST_IMG_DIR.glob("*.png"))

print(f"\n📊 Train images : {len(train_images)}")
print(f"📊 Test images  : {len(test_images)}")

In [ ]:
# ============================================================
# STEP 3A: Load labels and explore
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import seaborn as sns

# Load CSVs
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print("Train CSV columns:", df_train.columns.tolist())
print(df_train.head())
print(f"\nTrain samples : {len(df_train)}")
print(f"Test samples  : {len(df_test)}")
print(f"Unique labels : {df_train.iloc[:,1].nunique() if len(df_train.columns) > 1 else 'N/A'}")

## 🔍 Step 3 — Data Exploration

In [ ]:
# ============================================================
# STEP 3B: Normalize column names
# The CSV may have 'filename'/'image_name' and 'label'/'word'
# ============================================================
def normalize_df(df, img_dir):
    df = df.copy()
    cols = df.columns.tolist()

    # Detect filename column
    fname_col = next((c for c in cols if c.lower() in
                      ['filename','file_name','image','image_name','file']), cols[0])
    # Detect label column
    label_col = next((c for c in cols if c.lower() in
                      ['label','word','text','transcription','ground_truth']), cols[-1])

    df = df.rename(columns={fname_col: 'filename', label_col: 'label'})
    df['label'] = df['label'].astype(str).str.strip()

    # Build full image path
    def resolve_path(fname):
        p = img_dir / fname
        if p.exists(): return str(p)
        # Try with .jpg extension
        p2 = img_dir / (fname + '.jpg')
        if p2.exists(): return str(p2)
        # Search recursively
        matches = list(img_dir.glob(f"**/{fname}"))
        return str(matches[0]) if matches else str(p)

    df['filepath'] = df['filename'].apply(resolve_path)
    df['exists']   = df['filepath'].apply(os.path.exists)

    missing = (~df['exists']).sum()
    if missing > 0:
        print(f"⚠️  {missing} files not found — filtering them out")
        df = df[df['exists']].reset_index(drop=True)

    return df[['filename','label','filepath']]

df_train = normalize_df(df_train, TRAIN_IMG_DIR)
df_test  = normalize_df(df_test,  TEST_IMG_DIR)

print(f"\n✅ Train: {len(df_train)} | Test: {len(df_test)}")
print(df_train.head(3))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import seaborn as sns

# ── Sample images ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("📸 Sample Training Images", fontsize=15, fontweight='bold')
samples = df_train.sample(15, random_state=42).reset_index(drop=True)
for ax, (_, row) in zip(axes.flatten(), samples.iterrows()):
    img = Image.open(row['filepath']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"{row['label']}\n{img.size[0]}×{img.size[1]}",
                 fontsize=8, color='navy')
    ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "01_sample_images.png", dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("📊 RxHandBD Dataset Statistics", fontsize=14, fontweight='bold')

df_train['len'] = df_train['label'].str.len()
axes[0].hist(df_train['len'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Label character length')
axes[0].set_xlabel('characters')

top20 = df_train['label'].value_counts().head(20)
axes[1].barh(top20.index[::-1], top20.values[::-1], color='coral')
axes[1].set_title('Top-20 most frequent labels')

axes[2].pie([len(df_train), len(df_test)],
            labels=[f'Train ({len(df_train)})', f'Test ({len(df_test)})'],
            colors=['#4CAF50','#FF9800'], autopct='%1.1f%%', startangle=90)
axes[2].set_title('Train / Test split')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "02_dataset_stats.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Unique labels : {df_train['label'].nunique()}")
print(f"Avg label len : {df_train['len'].mean():.1f} chars")


## 🤖 Step 4 — Load TrOCR Processor & Model
`VisionEncoderDecoderModel` expects:
- `pixel_values`  → encoder  
- `labels`        → shifted right internally by the model to form decoder_input_ids  

**Do NOT pass `decoder_input_ids` manually** — the model handles teacher-forcing itself.


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

MODEL_NAME = "microsoft/trocr-base-handwritten"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📥 Loading processor …")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)

print("📥 Loading model …")
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# ── Required token-id settings for VisionEncoderDecoderModel ──────────────────
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.sep_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

# ── Generation hyperparams (used by model.generate) ───────────────────────────
model.config.max_length      = 64
model.config.no_repeat_ngram_size = 3
model.config.early_stopping  = True
model.config.length_penalty  = 2.0
model.config.num_beams       = 4

model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"\nTotal parameters     : {total_params:.1f} M")
print(f"Trainable parameters : {trainable_params:.1f} M")
print(f"Device               : {DEVICE}")


## 🔧 Step 5 — Apply LoRA (PEFT)
For `VisionEncoderDecoderModel`:
- LoRA should target **decoder** attention layers (the language model part).
- `task_type=SEQ_2_SEQ_LM` is the correct choice (not CAUSAL_LM).
- TrOCR's decoder is a `RobertaModel`; its attention projections are named `query`, `key`, `value`, `dense`.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# ── Inspect decoder linear layers ─────────────────────────────────────────────
print("Decoder linear layer names (sample):")
seen = set()
for name, mod in model.decoder.named_modules():
    if isinstance(mod, torch.nn.Linear):
        base = name.split('.')[-1]
        if base not in seen:
            seen.add(base)
            print(f"  {base}")


In [ ]:
# TrOCR decoder (RoBERTa-based) uses: query, key, value, dense
LORA_TARGET_MODULES = ["query", "value"]  # minimal set; add "key","dense" for more capacity

lora_cfg = LoraConfig(
    task_type     = TaskType.SEQ_2_SEQ_LM,   # correct for VisionEncoderDecoderModel
    r             = 16,
    lora_alpha    = 32,
    target_modules= LORA_TARGET_MODULES,
    lora_dropout  = 0.05,
    bias          = "none",
    inference_mode= False,
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print("✅ LoRA applied")


## 📚 Step 6 — Dataset Class & DataLoaders
### Key fixes vs. original
| Issue | Fix |
|---|---|
| Prompt text concatenated with label | Removed — TrOCR is **not** a prompted VLM |
| `decoder_input_ids` built manually | Not needed — `labels` is enough |
| Custom normalisation transforms | Use **TrOCRProcessor** for pixel_values |
| Complex prompt-masking logic | Deleted — just mask padding with `-100` |


In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import torchvision.transforms as T

# ── Augmentation (PIL-level, applied before processor) ────────────────────────
TRAIN_AUG = T.Compose([
    T.RandomRotation(degrees=3),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=0, shear=3),
])

class RxHandBDDataset(Dataset):
    """
    Returns:
        pixel_values  : FloatTensor [3, H, W]  — processor-normalised
        labels        : LongTensor  [seq_len]   — token ids; padding masked to -100
        label_text    : str                     — raw ground-truth word
        image_path    : str
    """
    MAX_LABEL_LEN = 64   # max tokens for a single prescription word

    def __init__(self, df, processor, augment=False):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Load image
        try:
            img = Image.open(row['filepath']).convert("RGB")
        except Exception:
            img = Image.new("RGB", (384, 384), (255, 255, 255))

        # 2. Optional augmentation (PIL only — before processor)
        if self.augment:
            img = TRAIN_AUG(img)

        # 3. Encode image → pixel_values via TrOCRProcessor
        #    The processor resizes to 384×384 and normalises with ViT stats.
        pixel_values = self.processor(
            images=img, return_tensors="pt"
        ).pixel_values.squeeze(0)   # [3, 384, 384]

        # 4. Tokenise label text
        label_text = row['label']
        encoding   = self.processor.tokenizer(
            label_text,
            padding   ="max_length",
            max_length=self.MAX_LABEL_LEN,
            truncation=True,
            return_tensors="pt",
        )
        labels = encoding.input_ids.squeeze(0).clone()   # [MAX_LABEL_LEN]

        # 5. Mask padding positions — model ignores these in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels"      : labels,
            "label_text"  : label_text,
            "image_path"  : row['filepath'],
        }


# ── Train / Val split ─────────────────────────────────────────────────────────
df_train_split, df_val_split = train_test_split(
    df_train, test_size=0.1, random_state=42
)
print(f"Train: {len(df_train_split)} | Val: {len(df_val_split)} | Test: {len(df_test)}")

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE  = 16    # reduce to 8 if OOM
GRAD_ACCUM  = 2     # effective batch = BATCH_SIZE × GRAD_ACCUM
NUM_WORKERS = 0     # 0 = stable on Colab/Kaggle

train_ds = RxHandBDDataset(df_train_split, processor, augment=True)
val_ds   = RxHandBDDataset(df_val_split,   processor, augment=False)
test_ds  = RxHandBDDataset(df_test,        processor, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


## 🏋️ Step 7 — Training Loop
### Key fixes vs. original
| Issue | Fix |
|---|---|
| `decoder_input_ids` passed manually (double-shift bug) | Removed — pass only `pixel_values` + `labels` |
| `bfloat16` autocast on T4 (T4 doesn't support bf16) | Use `fp16` autocast + `GradScaler` |
| Validation `model.generate(input_ids=...)` (text ids, not vision) | Use `model.generate(pixel_values=...)` |
| `tokenizer.decode(gen[input_ids.shape[1]:])` (strips wrong slice) | `processor.batch_decode(gen_ids, skip_special_tokens=True)` |


In [ ]:
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
from jiwer import wer as compute_wer, cer as compute_cer
from tqdm.auto import tqdm
import time

# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_EPOCHS    = 5
LR            = 5e-5       # lower LR works better for LoRA on encoder-decoder
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 100
MAX_GRAD_NORM = 1.0

# ── Detect bf16 support (A100 / H100); fall back to fp16 (T4/P100) ────────────
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
scaler    = None if USE_BF16 else GradScaler()
print(f"AMP dtype : {AMP_DTYPE}  |  GradScaler: {'off (bf16)' if USE_BF16 else 'on (fp16)'}")

optimizer   = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=total_steps,
)

print(f"Total optimiser steps : {total_steps}  (warmup {WARMUP_STEPS})")


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
history = {'epoch':[], 'train_loss':[], 'val_loss':[], 'val_cer':[], 'val_wer':[]}
best_val_loss  = float('inf')
best_ckpt_path = MODEL_DIR / "best_checkpoint"

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_losses = []
    optimizer.zero_grad()
    accum_count  = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]",
                leave=True, dynamic_ncols=True)

    for step, batch in enumerate(pbar):
        pixel_values = batch['pixel_values'].to(DEVICE)
        labels       = batch['labels'].to(DEVICE)

        try:
            with autocast(dtype=AMP_DTYPE):
                # ✅ CORRECT forward pass for VisionEncoderDecoderModel:
                #    pass pixel_values + labels only.
                #    The model internally right-shifts labels → decoder_input_ids.
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss    = outputs.loss / GRAD_ACCUM

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            accum_count += 1

            if accum_count == GRAD_ACCUM:
                if scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    MAX_GRAD_NORM
                )
                if scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                accum_count = 0

            train_losses.append(loss.item() * GRAD_ACCUM)
            pbar.set_postfix(loss=f"{train_losses[-1]:.4f}",
                             lr=f"{scheduler.get_last_lr()[0]:.2e}")

        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print(f"\n⚠️  OOM step {step} — skipping, resetting accum")
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                accum_count = 0
            else:
                raise

    # flush leftover gradients
    if accum_count > 0:
        if scaler:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), MAX_GRAD_NORM)
        if scaler:
            scaler.step(optimizer); scaler.update()
        else:
            optimizer.step()
        scheduler.step(); optimizer.zero_grad(); accum_count = 0

    avg_train = float(np.mean(train_losses))

    # ── Validation ─────────────────────────────────────────────────────────────
    model.eval()
    val_losses, val_preds, val_gts = [], [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]",
                          leave=False):
            pixel_values = batch['pixel_values'].to(DEVICE)
            labels       = batch['labels'].to(DEVICE)

            with autocast(dtype=AMP_DTYPE):
                outputs = model(pixel_values=pixel_values, labels=labels)
            val_losses.append(outputs.loss.item())

            # ✅ CORRECT generation: feed pixel_values, NOT text input_ids
            gen_ids = model.generate(
                pixel_values=pixel_values,
                max_new_tokens=64,
                num_beams=4,
                early_stopping=True,
            )
            # ✅ CORRECT decoding: processor handles special tokens & BOS/EOS
            preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
            val_preds.extend(preds)
            val_gts.extend(batch['label_text'])

    avg_val = float(np.mean(val_losses)) if val_losses else float('nan')

    # filter out empty ground truths before metric computation
    pairs    = [(g, p) for g, p in zip(val_gts, val_preds) if g.strip()]
    gts, pds = zip(*pairs) if pairs else ([], [])
    epoch_cer = compute_cer(list(gts), list(pds)) * 100 if gts else float('nan')
    epoch_wer = compute_wer(list(gts), list(pds)) * 100 if gts else float('nan')

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(avg_val)
    history['val_cer'].append(epoch_cer)
    history['val_wer'].append(epoch_wer)

    print(f"\n📈 Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train {avg_train:.4f} | Val {avg_val:.4f} | "
          f"CER {epoch_cer:.1f}% | WER {epoch_wer:.1f}% | "
          f"{time.time()-t0:.0f}s")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(str(best_ckpt_path))
        processor.save_pretrained(str(best_ckpt_path))
        print(f"   💾 Best checkpoint saved (val_loss={best_val_loss:.4f})")

    torch.cuda.empty_cache()

print("\n🏁 Training complete!")


## 📊 Step 8 — Training Curves

In [ ]:
epochs = history['epoch']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training Curves — TrOCR LoRA Fine-Tuning", fontsize=14, fontweight='bold')

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-s', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch')

axes[1].plot(epochs, history['val_cer'], 'g-o')
axes[1].set_title('Val CER (%) ↓'); axes[1].set_xlabel('epoch')

axes[2].plot(epochs, history['val_wer'], 'm-o')
axes[2].set_title('Val WER (%) ↓'); axes[2].set_xlabel('epoch')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "03_training_curves.png", dpi=120, bbox_inches='tight')
plt.show()


## 🧪 Step 9 — Full Evaluation on Test Set

In [ ]:
model.eval()
predictions, ground_truths, image_paths = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        pixel_values = batch['pixel_values'].to(DEVICE)

        with autocast(dtype=AMP_DTYPE):
            gen_ids = model.generate(
                pixel_values=pixel_values,
                max_new_tokens=64,
                num_beams=4,
                early_stopping=True,
            )

        preds = processor.batch_decode(gen_ids, skip_special_tokens=True)
        predictions.extend(preds)
        ground_truths.extend(batch['label_text'])
        image_paths.extend(batch['image_path'])

print(f"Evaluated {len(predictions)} samples")


In [ ]:
from collections import Counter

# filter empty ground truths
pairs    = [(g, p) for g, p in zip(ground_truths, predictions) if g.strip()]
gts, pds = zip(*pairs) if pairs else ([], [])

exact_match  = sum(g.lower().strip() == p.lower().strip() for g, p in zip(gts, pds))
accuracy     = exact_match / len(gts) * 100 if gts else 0
wer_score    = compute_wer(list(gts), list(pds)) * 100 if gts else 0
cer_score    = compute_cer(list(gts), list(pds)) * 100 if gts else 0
partial_acc  = sum(
    g.lower().strip() in p.lower() or p.lower().strip() in g.lower()
    for g, p in zip(gts, pds)
) / len(gts) * 100 if gts else 0

print("=" * 50)
print("📊 TEST RESULTS")
print("=" * 50)
print(f"  Samples         : {len(predictions)}")
print(f"  Exact Match     : {accuracy:.2f}%")
print(f"  Partial Match   : {partial_acc:.2f}%")
print(f"  WER             : {wer_score:.2f}%")
print(f"  CER             : {cer_score:.2f}%")
print("=" * 50)

# Sample predictions
print("\n📋 Sample predictions (first 10):")
print(f"{'Ground Truth':<25} | {'Prediction':<25} | Match")
print("-" * 60)
for g, p in zip(list(gts)[:10], list(pds)[:10]):
    icon = "✅" if g.lower().strip() == p.lower().strip() else "❌"
    print(f"{g:<25} | {p:<25} | {icon}")


## 🖼️ Step 10 — Prediction Visualisation

In [ ]:
import matplotlib.patches as mpatches

N_SHOW = min(16, len(predictions))
n_cols = 4
n_rows = (N_SHOW + n_cols - 1) // n_cols

fig = plt.figure(figsize=(22, n_rows * 5))
fig.suptitle("TrOCR Fine-Tuned — Prescription OCR Predictions",
             fontsize=14, fontweight='bold')

for i in range(N_SHOW):
    ax = fig.add_subplot(n_rows, n_cols, i + 1)
    try:
        ax.imshow(Image.open(image_paths[i]).convert('RGB'))
    except Exception:
        ax.imshow(np.ones((64, 256, 3), dtype=np.uint8) * 200)
    gt, pred  = ground_truths[i], predictions[i]
    ok        = gt.lower().strip() == pred.lower().strip()
    color     = '#4CAF50' if ok else '#F44336'
    for sp in ax.spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(3)
    ax.set_title(f"{'✅' if ok else '❌'} GT: {gt}\nPred: {pred or '(empty)'}",
                 fontsize=8, color='darkgreen' if ok else 'darkred', fontweight='bold')
    ax.axis('off')

fig.legend(
    handles=[mpatches.Patch(color='#4CAF50', label='Correct'),
             mpatches.Patch(color='#F44336', label='Incorrect')],
    loc='lower center', ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.01)
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "04_predictions.png", dpi=120, bbox_inches='tight')
plt.show()


## 💾 Step 11 — Save Model & Report

In [ ]:
import json, itertools

# LoRA adapter weights (small)
ADAPTER_PATH = MODEL_DIR / "lora_adapter"
ADAPTER_PATH.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_PATH))
processor.save_pretrained(str(ADAPTER_PATH))
print("Adapter saved to:", ADAPTER_PATH)

# Merged model (optional — larger, but inference without PEFT)
try:
    merged = model.merge_and_unload()
    MERGED_PATH = MODEL_DIR / "merged_model"
    MERGED_PATH.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGED_PATH), safe_serialization=True)
    processor.save_pretrained(str(MERGED_PATH))
    total_gb = sum(f.stat().st_size for f in MERGED_PATH.iterdir()) / 1e9
    print(f"Merged model saved ({total_gb:.2f} GB)")
except Exception as e:
    print(f"Merge skipped: {e}")

# Training report
report = dict(
    model=MODEL_NAME, dataset="RxHandBD",
    fine_tuning="LoRA", lora_r=16, lora_alpha=32,
    target_modules=LORA_TARGET_MODULES,
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, lr=LR,
    train_samples=len(df_train_split), val_samples=len(df_val_split),
    test_samples=len(df_test),
    metrics=dict(exact_match_pct=round(accuracy,2),
                 partial_match_pct=round(partial_acc,2),
                 WER_pct=round(wer_score,2),
                 CER_pct=round(cer_score,2)),
    training_history=history,
)
with open(RESULTS_DIR / "training_report.json", "w") as f:
    json.dump(report, f, indent=2)

import pandas as pd
pd.DataFrame(dict(image_path=image_paths,
                  ground_truth=ground_truths,
                  prediction=predictions,
                  exact_match=[g.lower().strip()==p.lower().strip()
                               for g,p in zip(ground_truths, predictions)])
).to_csv(RESULTS_DIR / "predictions.csv", index=False)

print("\n📂 Output files:")
for f in sorted(itertools.chain(
        RESULTS_DIR.glob("*.png"),
        RESULTS_DIR.glob("*.json"),
        RESULTS_DIR.glob("*.csv"))):
    print(f"  {f.name}")
print("\n🎉 Pipeline complete!")


## 🔄 Step 12 — Single-Image Inference

In [ ]:
# Reload best checkpoint (if needed)
# from peft import PeftModel
# base  = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
# model = PeftModel.from_pretrained(base, str(ADAPTER_PATH))
# processor = TrOCRProcessor.from_pretrained(str(ADAPTER_PATH))
# model.to(DEVICE).eval()

def predict(image_path, model=model, processor=processor, device=DEVICE):
    """Run TrOCR inference on a single prescription image."""
    img = Image.open(image_path).convert("RGB")
    pv  = processor(images=img, return_tensors="pt").pixel_values.to(device)
    model.eval()
    with torch.no_grad():
        ids = model.generate(pv, max_new_tokens=64, num_beams=4, early_stopping=True)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

# Demo
sample_path = df_test['filepath'].iloc[0]
sample_gt   = df_test['label'].iloc[0]
pred        = predict(sample_path)
print(f"Image      : {sample_path}")
print(f"Ground Truth : {sample_gt}")
print(f"Prediction   : {pred}")
print(f"Match        : {'✅' if sample_gt.lower()==pred.lower() else '❌'}")
